In [9]:
# Parameters
Month = 2
Year = 2569


In [10]:
import time
import pandas as pd
import os
from pathlib import Path
current_dir = Path.cwd()
# Month =2
# Year = 2569

thai_months = ["", "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
mytext = f"{thai_months[Month]} {str(Year)[-2:]}"
excel_file_path1 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด บ_{mytext}_.xlsx"
excel_file_path2 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด บ1_{mytext}_.xlsx"
excel_file_path3 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด น_{mytext}_.xlsx"

excel_file_path3

WindowsPath('c:/Users/User/Documents/GitHub/AutomationCPI/01-database/ราคากลาง/ไฟล์ราคากลาง ชุด น_ก.พ. 69_.xlsx')

In [11]:
def center(excel_file_path):
    try:
        # 1. อ่านไฟล์แบบไม่เอา Header (header=None) เพื่อจัดการหัวตารางเอง
        print(f"กำลังอ่านข้อมูลจากไฟล์ '{excel_file_path}'...")
        all_sheets_dict = pd.read_excel(excel_file_path, sheet_name=None, header=None)
        print(f"อ่านไฟล์เสร็จสิ้น พบทั้งหมด {len(all_sheets_dict)} ชีท")

    except Exception as e:
        print(f"เกิดข้อผิดพลาดในการเปิดไฟล์: {e}")
        all_sheets_dict = {}

    def center_logic(df_raw):
        # ลบแถวและคอลัมน์ที่เป็นค่าว่างทั้งหมดทิ้งก่อน เพื่อเช็คเนื้อหาจริง
        df_raw = df_raw.dropna(how='all').dropna(axis=1, how='all')
        
        # --- ตรรกะข้ามชีทเปล่า ---
        # ถ้าชีทไม่มีข้อมูล หรือมีจำนวนแถวน้อยเกินกว่าจะเป็นตารางราคา (ต้องมี Header 2 แถว + ข้อมูล)
        if df_raw.empty or len(df_raw) < 3:
            return pd.DataFrame()

        # --- ตรรกะเดิมของคุณ ---
        try:
            row_store = df_raw.iloc[0].copy()
            row_sub_header = df_raw.iloc[1].copy()
            row_store = row_store.ffill()
            
            df_data = df_raw.iloc[2:].reset_index(drop=True)
            col_code_idx = 0
            col_name_idx = 1
            
            dfs_list = []
            unique_stores = row_store[2:].dropna().unique()
            
            for store_name in unique_stores:
                if "ธ.ค." in str(store_name): continue
                
                store_indices = row_store[row_store == store_name].index
                idx_normal, idx_promo, idx_cond = None, None, None
                
                for idx in store_indices:
                    col_name = str(row_sub_header[idx])
                    if "ราคาปกติ" in col_name: idx_normal = idx
                    elif "ราคาโปร" in col_name: idx_promo = idx
                    elif "ภาวะ" in col_name: idx_cond = idx
                
                if idx_normal is not None:
                    temp_df = pd.DataFrame()
                    temp_df['รหัสรายการ\nCOMMODITY_CODE\n'] = df_data[col_code_idx]
                    temp_df['ชื่อรายการ'] = df_data[col_name_idx]
                    temp_df['ประเภทร้านค้า'] = store_name
                    temp_df['ราคาปกติ'] = df_data[idx_normal]
                    temp_df['ราคาโปร'] = df_data[idx_promo] if idx_promo is not None else None
                    temp_df['ภาวะ'] = df_data[idx_cond] if idx_cond is not None else None
                    dfs_list.append(temp_df)
            
            if not dfs_list: return pd.DataFrame()

            df_final = pd.concat(dfs_list, ignore_index=True)
            df_final = df_final.dropna(subset=['ราคาปกติ', 'ราคาโปร', 'ภาวะ'], how='all')
            return df_final
        except Exception:
            # กรณีโครงสร้างชีทผิดเพี้ยนจนรัน logic ไม่ได้ ให้คืนค่า DF เปล่าเพื่อข้ามชีทนี้
            return pd.DataFrame()

    # Loop รวมข้อมูลจากทุก Sheet
    list_of_dfs = []
    for name, df in all_sheets_dict.items():
        # ตรวจสอบเบื้องต้น ถ้าชีทว่าง (ไม่มีค่าใดๆ เลย) ให้ข้ามทันทีไม่ต้องเข้าฟังก์ชัน logic
        if df.empty or df.dropna(how='all').empty:
            # print(f"⏩ ข้ามชีทเปล่า: {name}")
            continue
            
        dcen = center_logic(df)
        if not dcen.empty:
            list_of_dfs.append(dcen)

    if list_of_dfs:
        allcenter = pd.concat(list_of_dfs, ignore_index=True)
        print(f"✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด {len(allcenter)} แถว")
    else:
        allcenter = pd.DataFrame()
        print("⚠️ ไม่พบข้อมูลที่ใช้งานได้ในไฟล์นี้")
        
    return allcenter

In [12]:
cen1 = center(excel_file_path1)
cen2 = center(excel_file_path2)
cen3 = center(excel_file_path3)

all_centers = pd.concat([cen1, cen2, cen3], ignore_index=True)

กำลังอ่านข้อมูลจากไฟล์ 'c:\Users\User\Documents\GitHub\AutomationCPI\01-database\ราคากลาง\ไฟล์ราคากลาง ชุด บ_ก.พ. 69_.xlsx'...
อ่านไฟล์เสร็จสิ้น พบทั้งหมด 62 ชีท
✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด 1257 แถว
กำลังอ่านข้อมูลจากไฟล์ 'c:\Users\User\Documents\GitHub\AutomationCPI\01-database\ราคากลาง\ไฟล์ราคากลาง ชุด บ1_ก.พ. 69_.xlsx'...
อ่านไฟล์เสร็จสิ้น พบทั้งหมด 17 ชีท
✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด 184 แถว
กำลังอ่านข้อมูลจากไฟล์ 'c:\Users\User\Documents\GitHub\AutomationCPI\01-database\ราคากลาง\ไฟล์ราคากลาง ชุด น_ก.พ. 69_.xlsx'...
อ่านไฟล์เสร็จสิ้น พบทั้งหมด 23 ชีท
✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด 46 แถว


In [13]:
all_centers

,รหัสรายการ\nCOMMODITY_CODE\n,ชื่อรายการ,ประเภทร้านค้า,ราคาปกติ,ราคาโปร,ภาวะ
0,1112104111020180,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 150 กรัม ตรา...,Lotus,16.5,NaN,NaN
1,1112104111030401,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 500 กรัม ตร...,Lotus,37,35,โปรจาก 37
2,1112104111010180,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 1 ก.ก. ตราครั...,Lotus,58,NaN,NaN
3,1112104111030101,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 500 กรัม ตรา...,Lotus,44,NaN,NaN
4,1112104111050101,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 120 กรัม ตรา...,Lotus,18,NaN,NaN
...,...,...,...,...,...,...
1482,NaN,กางเกงเก็บกระชับ รุ่น WG5041 สีเบจ,Central,1750,1487.5,NaN
1483,NaN,กางเกงชั้นใน ผู้หญิง เบสิค ทรงเต็มตัว - สีช๊อค...,Central,190,100,NaN
1484,NaN,กางเกงชั้นใน ผู้หญิง Smooth Sensation Hipster ...,Central,690,552,NaN
1485,NaN,กางเกงชั้นใน ผู้หญิง Sculpt Eclipse Midi-P 00A...,Central,690,552,NaN


In [14]:
# df_map = pd.read_excel('mapping_shop.xlsx')
# df_melted = df_map.melt(id_vars=['new_name'], value_name='old_name')
# df_melted = df_melted.dropna(subset=['old_name'])
# mapping_dict = dict(zip(df_melted['old_name'], df_melted['new_name']))
# mapping_dict

In [15]:
# หมด 1
# allcenter_clean = all_centers.copy()
# allcenter_clean['ประเภทร้านค้า'] = allcenter_clean['ประเภทร้านค้า'].replace({'BigC':'Big C','Lotus':'Tesco Lotus','7 - Eleven':'Seven Eleven','แม็คโคร':'Makro','CP Freshmart':'CP Freshmart'})

# df_map = pd.read_excel('mapping_shop.xlsx')
# df_melted = df_map.melt(id_vars=['new_name'], value_name='old_name')
# df_melted = df_melted.dropna(subset=['old_name'])
# mapping_dict = dict(zip(df_melted['old_name'], df_melted['new_name']))

# # 5. นำไปใช้งานกับ DataFrame หลัก
# allcenter_clean = all_centers.copy()
# allcenter_clean['ประเภทร้านค้า'] = allcenter_clean['ประเภทร้านค้า'].replace(mapping_dict)


# 1. อ่านไฟล์ Excel
df_map = pd.read_excel('shop_mapping2.xlsx')

def classify_advanced(shop_name):
    if not isinstance(shop_name, str):
        return "ร้านอื่นๆ"
    
    # วนลูปเช็คตามลำดับแถวใน Excel (สำคัญมาก: บนลงล่าง)
    for _, row in df_map.iterrows():
        target = row['Result']
        
        # เตรียม Primary Keywords (แยก comma)
        primaries = [k.strip() for k in str(row['Primary Keywords']).split(',') if k.strip()]
        
        # เตรียม Sub-Keywords (ถ้ามี)
        subs = [k.strip() for k in str(row['Sub Keywords']).split(',') if k.strip() != 'nan']

        # เช็ค Primary: ต้องเจออย่างน้อย 1 คำ
        if any(p in shop_name for p in primaries):
            # ถ้ามี Sub-Keywords: ต้องเจออย่างน้อย 1 คำในนี้ด้วย
            if subs:
                if any(s in shop_name for s in subs):
                    return target
                else:
                    continue # ถ้าไม่เจอ Sub ให้ไปเช็คกฎข้อถัดไป
            
            # ถ้าไม่มี Sub-Keywords แสดงว่าเจอ Primary ก็จบเลย
            return target
            
    return "ไม่ทราบแหล่ง"

# 3. นำไปใช้งาน
allcenter_clean = all_centers.copy()
# เปลี่ยนเป็นชื่อคอลัมน์ที่ถูกต้อง
allcenter_clean['ประเภทร้านค้า'] = allcenter_clean['ประเภทร้านค้า'].apply(classify_advanced)

allcenter_clean = allcenter_clean.rename(columns={'รหัสรายการ\nCOMMODITY_CODE\n':'COMMODITY_CODE','ชื่อรายการ':'DESCRIPTION'})
allcenter_clean = allcenter_clean[['COMMODITY_CODE','DESCRIPTION', 'ประเภทร้านค้า', 'ราคาปกติ','ราคาโปร','ภาวะ']]
allcenter_clean['COMMODITY_CODE'] = allcenter_clean['COMMODITY_CODE'].astype(str).str.zfill(16)
type(allcenter_clean['COMMODITY_CODE'][1])


# Step 1: Convert price columns to numeric, coercing errors to NaN
allcenter_clean['ราคาปกติ'] = pd.to_numeric(
    allcenter_clean['ราคาปกติ'], 
    errors='coerce'
)
allcenter_clean['ราคาโปร'] = pd.to_numeric(
    allcenter_clean['ราคาโปร'], 
    errors='coerce'
)
# Element-wise minimum for each row (axis=1)
allcenter_clean['ราคากลาง'] = allcenter_clean[['ราคาปกติ', 'ราคาโปร']].min(axis=1)
allcenter_clean = allcenter_clean[['COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคากลาง','ภาวะ']]

# allcenter_clean_outstock-----------------------------------------------------------------
allcenter_clean_outstock = allcenter_clean[allcenter_clean['ภาวะ']=='หมด'].copy()
allcenter_clean_outstock['ราคาปกติ'] = ""
allcenter_clean_outstock['ราคาโปร'] = ""
allcenter_clean_outstock = allcenter_clean_outstock[['COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคาปกติ', 'ราคาโปร','ภาวะ']]

# output_filename = filename[:-2] + dname +'_สินค้าหมด.xlsx'

# # กำหนดความกว้างแบบ Manual (อยากได้กว้างเท่าไหร่ใส่ตรงนี้)
# # หน่วยเป็นจำนวนตัวอักษรโดยประมาณ
# custom_widths = {
#     'COMMODITY_CODE': 20,    # รหัส 16 หลัก เผื่อไว้นิดหน่อย
#     'DESCRIPTION': 40,       # ชื่อสินค้าอาจจะยาว
#     'ประเภทร้านค้า': 15,
#     'ราคาปกติ': 12,
#     'ราคาโปร': 12,
#     # คอลัมน์ไหนไม่ใส่ในนี้ จะถูกคำนวณ Auto
# }
# with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:    
#     # 1. เขียนข้อมูลลง Sheet
#     sheet_name = 'Sheet1'
#     allcenter_clean_outstock.to_excel(writer, sheet_name=sheet_name, index=False)    
#     # 2. เข้าถึง object ของ worksheet
#     worksheet = writer.sheets[sheet_name]    
#     # 3. วนลูปทุกคอลัมน์
#     for i, col in enumerate(allcenter_clean_outstock.columns):        
#         # --- เช็คก่อนว่าคอลัมน์นี้เรากำหนดความกว้างเองไว้ไหม? ---
#         if col in custom_widths:
#             # ถ้ามีใน dict ให้ใช้ค่าที่กำหนดเลย
#             col_width = custom_widths[col]            
#         else:
#             # --- ถ้าไม่มี ให้คำนวณ Auto-fit ตามข้อมูลจริง ---            
#             # หาความยาวสูงสุดของข้อมูล
#             max_data_len = allcenter_clean_outstock[col].astype(str).map(len).max()
#             if pd.isna(max_data_len): 
#                 max_data_len = 0            
#             # หาความยาวของ Header
#             header_len = len(str(col))            
#             # เลือกค่ามากสุด + padding นิดหน่อย
#             col_width = max(max_data_len, header_len) + 2        
#         # สั่ง set ความกว้าง (XlsxWriter ใช้ set_column(first_col, last_col, width))
#         worksheet.set_column(i, i, col_width)
# # allcenter_clean_outstock-----------------------------------------------------------------

allcenter_clean
# allcenter[allcenter['ภาวะ']=='หมด']
# output_file2 = filename[:-2] + mytext + '_checkราคากลาง.xlsx'
allcenter_clean['Online'] = 'Online'
allcenter_clean = allcenter_clean.sort_values(by='COMMODITY_CODE')
# allcenter_clean.to_excel(output_file2)
allcenter_clean.to_excel(current_dir.parent / "01-database" / "ราคากลาง" / f"Concat_ราคากลาง_{mytext}_ชุดรายเดือน.xlsx", index=False)

In [23]:
allcenter_clean.columns
import pandas as pd

# กำหนดรายชื่อคอลัมน์
columns = ['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'ราคากลาง', 'ภาวะ', 'Online']

# สร้าง DataFrame เปล่า
allcenter_cleandf = pd.DataFrame(columns=columns)
allcenter_cleandf 

,COMMODITY_CODE,DESCRIPTION,ประเภทร้านค้า,ราคากลาง,ภาวะ,Online
